# Phase 1: Forward BCE Math & Numerical Stability

In this section, we will implement the **Binary Cross-Entropy (BCE)** loss function. BCE is used for binary classification because it heavily penalizes predictions that are confidently wrong.

## 1. The Intuition Behind BCE

The mathematical formula for calculating BCE over $N$ examples is:
$$E = -\frac{1}{N} \sum_{n=1}^{N} [y_n \log(p_n) + (1 - y_n) \log(1 - p_n)]$$

- $y_n$ is the true label (0 for Benign, 1 for Malignant).
- $p_n$ is our predicted probability that the tumor is Malignant.

### Why the negative sign?
Probabilities are always between 0 and 1. The logarithm of any value between 0 and 1 is a negative number! Since our goal is to minimize "Loss" (and we want a positive number to represent positive "cost"), we add a negative sign in front to flip the result back to positive.

### How it works
Remember that for any single example, $y_n$ can only be 0 or 1.
- If $y_n = 1$, the right half of the sum `(1 - 1)*...` disappears. The formula becomes just $- \log(p_n)$. If we predict $p_n$ close to 1, loss approaches 0. If we predict close to 0, the loss skyrockets.
- If $y_n = 0$, the left half disappears. The formula becomes just $- \log(1 - p_n)$. Similar logic applies.

* To undestand this deeply **visit the following documentation**: https://app.notion.com/p/jvalenci/Binary-Cross-entropy-Loss-3749d52658e080458d1be1b193770618

In [1]:
import numpy as np

def naive_bce(y_true, y_pred):
    """
    Calculates BCE directly from the formula without protections.
    """
    # y_true.shape[0] gives us the number of samples in the batch ( the rows in the input data )
    N = y_true.shape[0]
    return - (1 / N) * np.sum(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))

In [2]:

# Let's test this with a confidently wrong prediction.
# Example target: 1 (Malignant), our prediction is 0.0 (Extremely confident it's Benign)
y_t = np.array([1])
y_p = np.array([0.0])

# This will cause a numpy RuntimeWarning: divide by zero encountered in log
loss = naive_bce(y_t, y_p)
print(f"Loss: {loss}")

Loss: inf


/var/folders/lt/0wgz0hyd67z6zxh62hn6qff40000gn/T/ipykernel_16721/316868457.py:9: RuntimeWarning: divide by zero encountered in log
  return - (1 / N) * np.sum(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))


The naive version above outputs `inf` (infinity) and causes a `RuntimeWarning: divide by zero encountered in log` because `log(0)` is undefined. This happens if our network confidently outputs exactly **0.0** or **1.0**.

### Numerical Stability (The Fix)
To avoid infinite values and crashes, we use `np.clip` to enforce that predictions never mathematically hit 0.0 or 1.0. We clip them to a very small range like `[1e-15, 1 - 1e-15]`.

In [3]:
def stable_bce(y_true, y_pred):
    """
    Calculates BCE with numerical stability logic to handle log(0) issues.
    """
    # Clip y_pred between epsilon and 1-epsilon
    # This ensures that we never take log(0) which would be undefined and cause numerical issues.
    #? epsilon is a very small number
    # clip function will set any value in y_pred that is less than epsilon to epsilon
    epsilon = 1e-15
    y_pred_clipped = np.clip(y_pred, epsilon, 1 - epsilon)
    
    N = y_true.shape[0]
    return - (1 / N) * np.sum(y_true * np.log(y_pred_clipped) + (1 - y_true) * np.log(1 - y_pred_clipped))


In [4]:

# Let's test the same "dangerous" scenario again:
stable_loss = stable_bce(y_t, y_p)
print(f"Successfully calculated stable loss: {stable_loss:.5f}")

Successfully calculated stable loss: 34.53878


# Phase 2: BCE Derivative (Backward Pass)

During backpropagation, we need to calculate how much the neural network's predictions contributed to the overall error. This requires calculating the derivative of the BCE loss with respect to the predictions $p_n$, denoted mathematically as: $\frac{\partial E}{\partial p_n}$.

### The Calculus
Let's look at the BCE formula for a single example (ignoring the average $1/N$ for a moment):
$E = -[y \log(p) + (1 - y) \log(1 - p)]$

Using the basic derivative rule for natural logarithms ($\frac{d}{dx}\log(x) = \frac{1}{x}$) and the chain rule:
$$\frac{\partial E}{\partial p} = - \left( y \cdot \frac{1}{p} + (1 - y) \cdot \frac{1}{1-p} \cdot (-1) \right)$$
$$\frac{\partial E}{\partial p} = -\frac{y}{p} + \frac{1 - y}{1 - p}$$

Averaged over a batch of $N$ examples, the vectorized gradient formula becomes:
$$\frac{\partial E}{\partial p} = \frac{1}{N} \left( \frac{1 - y}{1 - p} - \frac{y}{p} \right)$$

<details>
<summary><span style="color:red"><b>Let's break down the exact algebra, step-by-step</b></span></summary>

Let's break down the exact algebra, step-by-step, showing exactly how those signs interact to give you that final positive fraction.

### **Step 1: Identify the two "chunks" inside the parentheses**

Let's look at the original equation. Inside the big parentheses, there are two distinct parts being added together:


$$\frac{\partial E}{\partial p} = - \left( \underbrace{y \cdot \frac{1}{p}}_{\text{Chunk 1}} + \underbrace{(1 - y) \cdot \frac{1}{1 - p} \cdot (-1)}_{\text{Chunk 2}} \right)$$

### **Step 2: Clean up Chunk 2**

Let's look only at Chunk 2. You have a fraction being multiplied by a $(-1)$ at the very end.
When you multiply anything by $-1$, you just put a negative sign in front of it.


$$\text{Chunk 2 becomes: } - \frac{1 - y}{1 - p}$$

### **Step 3: Rewrite the equation with the cleaned-up chunks**

Now, let's put our simplified Chunk 1 and Chunk 2 back into the main parentheses:


$$\frac{\partial E}{\partial p} = - \left( \frac{y}{p} - \frac{1 - y}{1 - p} \right)$$


Notice how the `+` sign from the original equation turned into a `-` sign because Chunk 2 became negative.

### **Step 4: Distribute the outside negative sign**

We still have that big negative sign waiting on the far left, completely outside the parentheses. In algebra, a negative sign directly in front of parentheses means you must multiply *everything* inside by $-1$.

Let's distribute it to both chunks:

* **Applying it to Chunk 1:** $-1 \cdot \left( \frac{y}{p} \right) \rightarrow \mathbf{-\frac{y}{p}}$
* **Applying it to Chunk 2:** $-1 \cdot \left( - \frac{1 - y}{1 - p} \right)$ ... *A negative times a negative equals a positive!* ... $\rightarrow \mathbf{+ \frac{1 - y}{1 - p}}$

### **The Final Result**

When you drop the parentheses and write those two final pieces together side-by-side, you get your final formula:


$$\frac{\partial E}{\partial p} = -\frac{y}{p} + \frac{1 - y}{1 - p}$$

The positive sign appears because you are multiplying the negative generated by the Chain Rule (inside) by the negative from the original BCE formula (outside).
</details>

**Important Note:** Just like the forward pass, this formula suffers from catastrophic divide-by-zero errors if our prediction $p$ mathematically hits exactly $0$ or $1$. We will use the same $1e-15$ `epsilon` clipping technique here to safeguard the backward pass.

In [5]:
def stable_bce_derivative(y_true, y_pred):
    """
    Calculates the derivative of BCE with respect to predictions (p),
    incorporating numerical stability to prevent division by zero.

    return: (the gradient dE/dp which is used in backpropagation to update model weights.)

    A gradient it's a vector of partail derivatives \
        (partial derivative is the rate of change of the loss with respect to a small change in the prediction p)
        so during backpropagation, we use this gradient to adjust the model's weights in the direction that reduces the loss.

    """

    # 1. Provide numeric stability
    epsilon = 1e-15
    y_pred_clipped = np.clip(y_pred, epsilon, 1 - epsilon)

    # 2. Get batch size
    N = y_true.shape[0]

    # 3. Calculate gradient: dE/dp = (1/N) * ((1-y)/(1-p) - y/p)
    term1 = (1 - y_true) / (1 - y_pred_clipped)
    term2 = y_true / y_pred_clipped

    gradient = (term1 - term2) / N

    return gradient


In [6]:

# Test the backward pass gradient calculation
print(f"Target (y_true):\t {y_t}")
print(f"Prediction (y_pred):\t {y_p}")

grad = stable_bce_derivative(y_t, y_p)
print(f"Gradient (dE/dp):\t {grad}")
print(f"Gradient Shape:\t\t {grad.shape}")

Target (y_true):	 [1]
Prediction (y_pred):	 [0.]
Gradient (dE/dp):	 [-1.e+15]
Gradient Shape:		 (1,)
